In [ ]:
"""
Signaling Game with Occasionally Observed World States
(Numba-optimized version)
"""

import numpy as np
from numba import njit
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from dataclasses import dataclass
from enum import IntEnum
import time
from datetime import datetime
import os
import csv

# ---- Game parameters ----
N = 4                  
P_OBSERVE = 1        
TIMESTEPS = 10_000_000  
N_RUNS = 100           

# ---- Variant selection ----
ACTION_SELECTION = "BP"
REINFORCEMENT = "no_gating"

# ---- What to run ----
RUN_SINGLE = True           
RUN_COMPARISON = False      
RUN_DUAL_THRESHOLD = False  
SAVE_PLOTS = True           
SHOW_PLOTS = False          
SHOW_FAILED_STRATEGIES = False  
RUN_SCALING_TABLE = True

class ActionSelectionEnum(IntEnum):
    LP = 0  
    BP = 1  
    SO = 2  

class ReinforcementEnum(IntEnum):
    NO_GATING = 0
    GATED = 1

def parse_action_selection(s: str) -> ActionSelectionEnum:
    mapping = {"LP": ActionSelectionEnum.LP, "BP": ActionSelectionEnum.BP, "SO": ActionSelectionEnum.SO}
    return mapping[s.upper()]

def parse_reinforcement(s: str) -> ReinforcementEnum:
    mapping = {"NO_GATING": ReinforcementEnum.NO_GATING, "GATED": ReinforcementEnum.GATED}
    return mapping[s.upper().replace("-", "_")]

@njit
def compute_normalized_entropy(probs):
    """Calculates entropy normalized by log(N) so it ranges from 0 to 1."""
    n = len(probs)
    if n <= 1: return 0.0
    ent = 0.0
    for p in probs:
        if p > 1e-12:
            ent -= p * np.log(p)
    return ent / np.log(n)

@njit
def compute_mixture_metrics(Q_sender, Q_recv_state):
    """
    1) State Mixture:       Avg entropy of P(signal|state)     [sender]
    2) Signal Mixture:      Avg entropy of P(state|signal)     [sender inverted]
    3) Recv State Mixture:  Avg entropy of P(action|state)     [receiver, state-only]
                            0 = knows exactly what to do; 1 = no idea
    """
    n = Q_sender.shape[0]
    
    # ── Sender strategy: P(sig | state) ──────────────────────────────────────
    sender_strat = np.zeros((n, n))
    for i in range(n):
        row_sum = Q_sender[i].sum()
        sender_strat[i] = Q_sender[i] / row_sum

    # 1. State Mixture — how mixed is each row of the sender strategy?
    sum_state_ent = 0.0
    for i in range(n):
        sum_state_ent += compute_normalized_entropy(sender_strat[i])
    avg_state_mixture = sum_state_ent / n

    # 2. Signal Mixture — invert to get P(state | signal), then measure entropy
    p_state_given_sig = np.zeros((n, n))   # [signal, state]
    for sig in range(n):
        col_sum = 0.0
        for st in range(n):
            col_sum += sender_strat[st, sig]
        if col_sum > 1e-12:
            for st in range(n):
                p_state_given_sig[sig, st] = sender_strat[st, sig] / col_sum
        else:
            for st in range(n):
                p_state_given_sig[sig, st] = 1.0 / n

    sum_sig_ent = 0.0
    for j in range(n):
        sum_sig_ent += compute_normalized_entropy(p_state_given_sig[j])
    avg_signal_mixture = sum_sig_ent / n

    # 3. Receiver State-Only Mixture — avg entropy of P(action | state)
    #    0 = perfectly decisive (knows what action to take given state)
    #    1 = completely uncertain
    sum_recv_state_ent = 0.0
    for i in range(n):
        row_sum = Q_recv_state[i].sum()
        recv_probs = Q_recv_state[i] / row_sum
        sum_recv_state_ent += compute_normalized_entropy(recv_probs)
    avg_recv_state_mixture = sum_recv_state_ent / n

    return avg_state_mixture, avg_signal_mixture, avg_recv_state_mixture

@njit
def choose_from_probs(probs):
    r = np.random.random()
    cumsum = 0.0
    for i in range(len(probs)):
        cumsum += probs[i]
        if r < cumsum: return i
    return len(probs) - 1

@njit
def run_single_simulation(n, p_observe, action_selection, reinforcement, timesteps):
    Q_sender = np.ones((n, n))
    Q_recv_signal = np.ones((n, n))
    Q_recv_state = np.ones((n, n))
    total_payoff = 0
    for t in range(timesteps):
        state = np.random.randint(0, n)
        sender_probs = Q_sender[state] / Q_sender[state].sum()
        signal = choose_from_probs(sender_probs)
        observe_state = np.random.random() < p_observe
        if observe_state:
            probs_state = Q_recv_state[state] / Q_recv_state[state].sum()
            probs_signal = Q_recv_signal[signal] / Q_recv_signal[signal].sum()
            rec_state = choose_from_probs(probs_state)
            rec_signal = choose_from_probs(probs_signal)
            if action_selection == 2: action = choose_from_probs(probs_state)
            elif action_selection == 0:
                pooled_probs = 0.5 * probs_state + 0.5 * probs_signal
                action = choose_from_probs(pooled_probs)
            else:
                pooled_Q = Q_recv_state[state] + Q_recv_signal[signal]
                action = choose_from_probs(pooled_Q / pooled_Q.sum())
            success = (action == state)
            if success:
                total_payoff += 1
                Q_sender[state, signal] += 1
                if reinforcement == 0:
                    Q_recv_state[state, action] += 1
                    Q_recv_signal[signal, action] += 1
                else:
                    if rec_state == action: Q_recv_state[state, action] += 1
                    if rec_signal == action: Q_recv_signal[signal, action] += 1
        else:
            probs_signal = Q_recv_signal[signal] / Q_recv_signal[signal].sum()
            action = choose_from_probs(probs_signal)
            success = (action == state)
            if success:
                total_payoff += 1
                Q_sender[state, signal] += 1
                Q_recv_signal[signal, action] += 1
    return Q_sender, Q_recv_signal, Q_recv_state, total_payoff

@njit
def compute_expected_payoff(n, p_observe, action_selection, Q_sender, Q_recv_signal, Q_recv_state):
    sender_strat = np.zeros((n, n))
    for i in range(n): sender_strat[i] = Q_sender[i] / Q_sender[i].sum()
    recv_signal_strat = np.zeros((n, n))
    recv_state_strat = np.zeros((n, n))
    for i in range(n):
        recv_signal_strat[i] = Q_recv_signal[i] / Q_recv_signal[i].sum()
        recv_state_strat[i] = Q_recv_state[i] / Q_recv_state[i].sum()
    exp_signal_only = 0.0
    for i in range(n):
        for j in range(n): exp_signal_only += sender_strat[i, j] * recv_signal_strat[j, i]
    exp_signal_only /= n
    exp_state_obs = 0.0
    for i in range(n):
        for j in range(n):
            p_sig = sender_strat[i, j]
            if action_selection == 2: p_correct = recv_state_strat[i, i]
            elif action_selection == 0: p_correct = 0.5 * recv_state_strat[i, i] + 0.5 * recv_signal_strat[j, i]
            else:
                Q_pooled = Q_recv_state[i] + Q_recv_signal[j]
                p_correct = Q_pooled[i] / Q_pooled.sum()
            exp_state_obs += p_sig * p_correct
    exp_state_obs /= n
    return (1 - p_observe) * exp_signal_only + p_observe * exp_state_obs

@njit
def run_multiple_simulations(n, p_observe, action_selection, reinforcement, timesteps, n_runs):
    final_expected_payoffs = np.zeros(n_runs)
    average_payoffs        = np.zeros(n_runs)
    state_mixtures         = np.zeros(n_runs)
    signal_mixtures        = np.zeros(n_runs)
    recv_state_mixtures    = np.zeros(n_runs)

    for run in range(n_runs):
        Q_sender, Q_recv_signal, Q_recv_state, total_payoff = run_single_simulation(
            n, p_observe, action_selection, reinforcement, timesteps
        )
        final_expected_payoffs[run] = compute_expected_payoff(
            n, p_observe, action_selection, Q_sender, Q_recv_signal, Q_recv_state
        )
        average_payoffs[run] = total_payoff / timesteps
        st_mix, sig_mix, rs_mix = compute_mixture_metrics(Q_sender, Q_recv_state)
        state_mixtures[run]      = st_mix
        signal_mixtures[run]     = sig_mix
        recv_state_mixtures[run] = rs_mix

    return final_expected_payoffs, average_payoffs, state_mixtures, signal_mixtures, recv_state_mixtures

@dataclass
class GameConfig:
    n: int = 2
    p_observe: float = 0.1
    action_selection: ActionSelectionEnum = ActionSelectionEnum.LP
    reinforcement: ReinforcementEnum = ReinforcementEnum.NO_GATING
    timesteps: int = 10000
    n_runs: int = 100
    def variant_name(self) -> str:
        a_names = {0: "LP", 1: "BP", 2: "SO"}
        r_names = {0: "no_gating", 1: "gated"}
        return f"{a_names[int(self.action_selection)]}, {r_names[int(self.reinforcement)]}"
    @classmethod
    def from_globals(cls):
        return cls(n=N, p_observe=P_OBSERVE, timesteps=TIMESTEPS, n_runs=N_RUNS,
                   action_selection=parse_action_selection(ACTION_SELECTION),
                   reinforcement=parse_reinforcement(REINFORCEMENT))

def run_experiments(config: GameConfig, verbose: bool = True) -> dict:
    if verbose: print(f"Running {config.n_runs} simulations...", flush=True)
    start = time.time()
    
    final_payoffs, avg_payoffs, st_mixes, sig_mixes, rs_mixes = run_multiple_simulations(
        config.n, config.p_observe, int(config.action_selection),
        int(config.reinforcement), config.timesteps, config.n_runs
    )
    
    if verbose: print(f"Completed in {time.time() - start:.2f}s", flush=True)
    
    return {
        'final_expected_payoffs':   final_payoffs,
        'average_payoffs':          avg_payoffs,
        'state_mixtures':           st_mixes,
        'signal_mixtures':          sig_mixes,
        'recv_state_mixtures':      rs_mixes,
        'success_rate':             np.mean(final_payoffs >= 0.875),
        'mean_final_expected':      np.mean(final_payoffs),
        'mean_average_payoff':      np.mean(avg_payoffs),
        'mean_state_mixture':       np.mean(st_mixes),
        'mean_signal_mixture':      np.mean(sig_mixes),
        'mean_recv_state_mixture':  np.mean(rs_mixes),
    }

def _fmt_timestep(t: int) -> str:
    exponent = round(np.log10(t))
    if 10 ** exponent == t: return f"1e{exponent}"
    return f"{t:,}"

def _save_cell_distribution_plot(final_payoffs, variant_name, n, timesteps, p_obs, n_runs,
                                  success_rate, mean_avg_payoff, st_mix, sig_mix, rs_mix, save_path):
    plt.style.use("default")
    fig, ax = plt.subplots(figsize=(8, 5), facecolor="white")
    ax.hist(final_payoffs, bins=20, density=True, alpha=0.7, color="steelblue", edgecolor="white")
    ax.axvline(x=0.875, color="red", linestyle="--", label=f"Threshold 0.875")
    ax.set_title(
        f"{variant_name} | n={n}, T={timesteps:,}, p={p_obs:.1f}\n"
        f"Success: {success_rate*100:.1f}% | Avg Payoff: {mean_avg_payoff:.3f}\n"
        f"State Mix: {st_mix:.3f} | Signal Mix: {sig_mix:.3f} | Recv-State Mix: {rs_mix:.3f}",
        fontsize=9
    )
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1.05)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.close()

def _n_runs_for_timestep(t: int) -> int:
    """
    10^2 – 10^6  plays/run  →  1 000 runs
    10^7         plays/run  →    100 runs
    """
    return 100 if t >= 10_000_000 else 1_000

def run_scaling_table(n=4, save_dir="scaling_results"):
    t_grid = [100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000]
    p_grid = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
    variants = [
        ("LP", "no_gating", 0, 0), ("LP", "gated", 0, 1),
        ("BP", "no_gating", 1, 0), ("BP", "gated", 1, 1),
        ("SO", "no_gating", 2, 0), ("SO", "gated", 2, 1)
    ]

    os.makedirs(save_dir, exist_ok=True)
    summary_path = os.path.join(save_dir, "all_metrics_summary.txt")

    # ── Column layout ────────────────────────────────────────────────────────
    # Each p-column holds: "Succ% Pay StMx SgMx RsMx"
    # Field widths: 5 4 4 4 4  →  total 21 chars + 1 space separator = 22
    COL_W   = 22          # total width of one p-column (including trailing space)
    PLAYS_W = 8           # width of the "Plays" label column

    def col_header(p):
        """Centered p-label, exactly COL_W wide."""
        label = f"p={p:.1f}"
        return label.center(COL_W - 1) + " "   # -1 for the "|" separator

    def col_header_sub():
        """Sub-header showing metric abbreviations, exactly COL_W wide."""
        sub = "Suc% Pay StMx SgMx RsMx"
        return sub.center(COL_W - 1) + " "

    def divider(n_p):
        return "-" * (PLAYS_W + 3 + n_p * (COL_W + 1))

    with open(summary_path, "w") as f_sum:
        f_sum.write(f"SCALING TABLE  n={n}  {datetime.now()}\n")
        f_sum.write("Metrics: Suc%=success rate  Pay=avg payoff  "
                    "StMx=state mixture  SgMx=signal mixture  RsMx=recv-state mixture\n"
                    "RsMx: 0=receiver knows exactly what to do, 1=no idea\n\n")

        for action_label, reinf_label, a_sel, reinf in variants:
            v_name = f"{action_label}, {reinf_label}"
            f_sum.write(f"{'='*80}\nVARIANT: {v_name}\n{'='*80}\n")

            # ── Header rows ──────────────────────────────────────────────────
            header1 = f"{'Plays':<{PLAYS_W}} | " + "| ".join(col_header(p) for p in p_grid)
            header2 = f"{'':>{PLAYS_W}} | " + "| ".join(col_header_sub() for _ in p_grid)
            div     = divider(len(p_grid))
            f_sum.write(header1 + "\n")
            f_sum.write(header2 + "\n")
            f_sum.write(div     + "\n")

            for t in t_grid:
                current_n_runs = _n_runs_for_timestep(t)
                row_str = f"{_fmt_timestep(t):<{PLAYS_W}} | "

                for p in p_grid:
                    res, avgs, st, sig, rs = run_multiple_simulations(
                        n, p, a_sel, reinf, t, current_n_runs
                    )

                    s_rate = np.mean(res >= 0.875) * 100
                    a_pay  = np.mean(avgs)
                    m_st   = np.mean(st)
                    m_sig  = np.mean(sig)
                    m_rs   = np.mean(rs)

                    # 5 metrics, each 4 chars wide (e.g. " 87%" "0.92" ".23 ")
                    cell = f"{s_rate:4.0f}% {a_pay:.2f} {m_st:.2f} {m_sig:.2f} {m_rs:.2f} "
                    row_str += cell + "| "

                    # ── Per-cell distribution plot ───────────────────────────
                    dist_dir = os.path.join(save_dir, "plots", v_name.replace(", ", "_"))
                    os.makedirs(dist_dir, exist_ok=True)
                    _save_cell_distribution_plot(
                        res, v_name, n, t, p, current_n_runs,
                        s_rate / 100, a_pay, m_st, m_sig, m_rs,
                        os.path.join(dist_dir, f"T{_fmt_timestep(t)}_p{p:.1f}.png")
                    )

                f_sum.write(row_str + "\n")

            f_sum.write("\n")

    print(f"Scaling table saved to: {summary_path}")

def main():
    # 1. Create a non-overriding session directory
    session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_dir = f"session_{session_id}"
    os.makedirs(base_dir, exist_ok=True)
    print(f"All results will be saved in: {base_dir}")

    # JIT Warmup
    print("Warming up JIT...", end=" ", flush=True)
    run_multiple_simulations(2, 0.1, 0, 0, 100, 2)
    print("Done.\n")

    if RUN_SINGLE:
        config = GameConfig.from_globals()
        results = run_experiments(config)
        print(f"\nResults for {config.variant_name()}:")
        print(f"  Success Rate:        {results['success_rate']*100:.1f}%")
        print(f"  Avg Payoff:          {results['mean_average_payoff']:.4f}")
        print(f"  State Mixture:       {results['mean_state_mixture']:.4f}")
        print(f"  Signal Mixture:      {results['mean_signal_mixture']:.4f}")
        print(f"  Recv-State Mixture:  {results['mean_recv_state_mixture']:.4f}")

    if RUN_SCALING_TABLE:
        print("\nRunning Scaling Tables (this may take a while)...")
        run_scaling_table(n=N, save_dir=os.path.join(base_dir, "scaling"))

if __name__ == "__main__":
    main()

All results will be saved in: session_20260422_222709
Warming up JIT... Done.

Running 100 simulations...
Completed in 357.50s

Results for BP, no_gating:
  Success Rate:    84.0%
  Avg Payoff:      0.9363
  State Mixture:   0.1230
  Signal Mixture:  0.1301
Running Scaling Tables (this may take a while)...
